# Cell 1: Import everything:


In [1]:
import torch
import numpy as np

from sklearn.model_selection import train_test_split
from transformers import BertModel, BertConfig
from torch.utils.data import DataLoader

# Data handeling
from src.data.dataset import MLMDataset, ClassificationDataset, HieraricalClassificationDataset
from src.data.data_tools import filter_taxonomy, fasta2pandas

#Vocab shit
from src.utils.vocab import Vocabulary, KmerVocabConstructor

# Preprocessing
from src.preprocessing.augmentation import SequenceModifier, IdentityStrategy, BaseStrategy
from src.preprocessing.tokenization import KmerStrategy
from src.preprocessing.padding import PEndStrategy
from src.preprocessing.truncation import TEndStrategy
from src.preprocessing.preprocessor import Preprocessor

# Model things
from src.model.backbone import Bertax, ModularBertax
from src.model.encoders import LabelEncoder
from src.model.heads import MLMHead, SingleClassHead, HierarchicalClassificationHead

# Training things
from src.train.trainers import MLMtrainer, ClassificationTrainer

c:\Users\Marcus\Documents\MasterProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
CONFIG = {
    "FILE_PATH": "data/raw.fasta",
    "SAVE_PATH": "pretrained_model.pt",
    "n_test": 10,
    "modification_probability": 0.05,
    "alphabet": ["A", "C", "G", "T"],
    "k": 3,
    "optimal_length": 200,

    # Training parameters
    "num_epochs": 1,
    "masking_percentage": 0.05,
    "batch_size": 128,
    "small_set": True,

    # Model configuration
    "num_layers": 10,
    "num_attention_heads": 4,
    "hidden_size": 256,
    "intermediate_size": 1024,  # 4 * hidden_size
    "dropout_rate": 0.05,
    "num_classes": 19,
    "mlm_dropout_rate": 0.1,

    #Classification 
    "target_labels": ["phylum", "class", "order"]
}

In [4]:

# Set up vocabulary
constructor = KmerVocabConstructor(k=CONFIG["k"], alphabet=CONFIG["alphabet"])
vocab = Vocabulary()
vocab.build_from_constructor(constructor, data=[])
vocab_path = "vocab.json"
vocab.save(vocab_path)

# Set up preprocessors
sequence_modifier = SequenceModifier(alphabet=CONFIG["alphabet"])
augmentation_strategy_train = BaseStrategy(
    modifier=sequence_modifier,
    alphabet=CONFIG["alphabet"],
    modification_probability=CONFIG["modification_probability"]
)

augmentation_strategy_val = IdentityStrategy(
    modifier=sequence_modifier,
    alphabet=CONFIG["alphabet"],
    modification_probability=0
)

tokenization_strategy = KmerStrategy(
    k=CONFIG["k"],
    padding_alphabet=CONFIG["alphabet"]
)

padding_strategy = PEndStrategy(
    optimal_length=CONFIG["optimal_length"]
)

truncation_strategy = TEndStrategy(
    optimal_length=CONFIG["optimal_length"]
)

preprocessor_train = Preprocessor(
    augmentation_strategy=augmentation_strategy_train,
    tokenization_strategy=tokenization_strategy,
    padding_strategy=padding_strategy,
    truncation_strategy=truncation_strategy,
    vocab=vocab,
)

preprocessor_val = Preprocessor(
    augmentation_strategy=augmentation_strategy_val,
    tokenization_strategy=tokenization_strategy,
    padding_strategy=padding_strategy,
    truncation_strategy=truncation_strategy,
    vocab=vocab,
)



In [12]:

##################################################################
## Data preparation ##############################################
##################################################################

all_data = fasta2pandas(CONFIG["FILE_PATH"])

#tmp: to use a smaller set for test if code compiles - nothing to be used in production
if CONFIG["small_set"]:
    all_data = all_data[:CONFIG["n_test"]]

target_labels = CONFIG["target_labels"]

# filter data for finetuning
filtered_data = filter_taxonomy(
    df = all_data,
    startAt =target_labels[0],
    endAt = target_labels[-1],
    phylumCertainty=True
    )

# 

class_sizes = [len(list(set(filtered_data[target_label]))) for target_label in target_labels]

label_encoders = {}
for target_label in target_labels:
    label_encoders[target_label] = LabelEncoder(list(set(filtered_data[target_label])))
    
# Pretraining datasplit based on unsplit data: 
pretrain_sequences, preval_sequences = train_test_split(
    all_data["sequence"],
    test_size = 0.1,
    random_state = 42
)

# Finetune datasplit based on filtered data:
finetrain_data, fineval_data = train_test_split(
    filtered_data,
    test_size = 0.1,
    random_state = 69
    )

print(f"Number of pre-training sequences: {len(pretrain_sequences)}")
print(f"Number of validation sequences: {len(preval_sequences)}")


Applying filters...
Filtering complete.

Handling DNA ambiguity codes...
Processing row 0...
Processing complete.
Number of pre-training sequences: 9
Number of validation sequences: 1


In [20]:
ids = range(9)


for id in ids:
    print("Id", id)
    for target_lvl, label_encoder in label_encoders.items():
        label = all_data[target_lvl][id]
        print(target_lvl, label, label_encoder.encode(label))
    print()

Id 0
phylum Ascomycota 1
class Eurotiomycetes 2
order Verrucariales 1

Id 1
phylum Ascomycota 1
class Dothideomycetes 0
order Stigmatodiscales 5

Id 2
phylum Rozellomycota 2
class Rozellomycota_cls_Incertae_sedis 3
order Rozellomycota_ord_Incertae_sedis 0

Id 3
phylum Basidiomycota 0
class Agaricomycetes 1
order Agaricales 3

Id 4
phylum Basidiomycota 0
class Agaricomycetes 1
order Sebacinales 6

Id 5
phylum Basidiomycota 0
class Agaricomycetes 1
order Sebacinales 6

Id 6
phylum Basidiomycota 0
class Agaricomycetes 1
order Thelephorales None

Id 7
phylum Fungi_phy_Incertae_sedis None
class Fungi_cls_Incertae_sedis None
order Fungi_ord_Incertae_sedis None

Id 8
phylum Basidiomycota 0
class Agaricomycetes 1
order Corticiales 2



In [5]:
print(label_encoders)

{'phylum': <src.model.encoders.LabelEncoder object at 0x2a02d8d70>, 'class': <src.model.encoders.LabelEncoder object at 0x2a448c440>, 'order': <src.model.encoders.LabelEncoder object at 0x2a44b5a00>}


In [21]:
##################################################################
## Setup Datasets ################################################
##################################################################

# Pretraining datasets
pretrain_dataset = MLMDataset(
    df = pretrain_sequences,
    preprocessor = preprocessor_train,
    masking_percentage = CONFIG["masking_percentage"]
)

preval_dataset = MLMDataset(
    df = preval_sequences,
    preprocessor = preprocessor_val,
    masking_percentage = CONFIG["masking_percentage"]
)

# Finetuning datasets
finetrain_dataset = HieraricalClassificationDataset(
    df = finetrain_data,
    preprocessor = preprocessor_train,
    label_encoders=label_encoders
    )

fineval_dataset = HieraricalClassificationDataset(
    df = fineval_data,
    preprocessor = preprocessor_val,
    label_encoders = label_encoders,
    )

##################################################################
## Setup Dataloader ##############################################
##################################################################

pretrain_loader = DataLoader(
    dataset = pretrain_dataset,
    batch_size = CONFIG["batch_size"],
    shuffle = True
)

preval_loader = DataLoader(
    dataset = preval_dataset,
    batch_size = CONFIG["batch_size"],
    shuffle = True
)

#data loaders
finetrain_loader = DataLoader(
    dataset=finetrain_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True
    )

fineval_loader = DataLoader(
    dataset=fineval_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True
    )



In [22]:
encoder_config = BertConfig(
    vocab_size = len(vocab),
    hidden_size = CONFIG["hidden_size"],
    num_hidden_layers = CONFIG["num_layers"],
    num_attention_heads = CONFIG["num_attention_heads"],
    intermediate_size = CONFIG["intermediate_size"],
    max_position_embeddings = CONFIG["optimal_length"] + 2,
    hidden_dropout_prob = CONFIG["dropout_rate"],
    attention_probs_dropout_prob = CONFIG["dropout_rate"]
)

encoder = BertModel(encoder_config)

mlm_head = MLMHead(
    in_features = CONFIG["hidden_size"],
    hidden_layer_size = CONFIG["hidden_size"] // 2,
    out_features = len(vocab),
    dropout_rate = CONFIG["mlm_dropout_rate"]
)


classification_head = HierarchicalClassificationHead(
    in_features=CONFIG["hidden_size"],
    class_sizes= class_sizes,
    dropout_rate=CONFIG["dropout_rate"]
    )

model = ModularBertax(
    encoder = encoder,
    mlm_head = mlm_head,
    classification_head = classification_head
)


# Eval model with single input


In [26]:
model.classifyMode()
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 4) Get a single item from your dataset
item = finetrain_dataset.__getitem__(idx=1)
input_ids = item["input_ids"].unsqueeze(0).to(device)       # [1, seq_len]
attention_mask = item["attention_mask"].unsqueeze(0).to(device)
lvl_output = item["output"]

# 5) Inference
with torch.no_grad():
    logits = model(input_ids, attention_mask)  # shape: [1, num_classes]
    
    print("\n--- Input Information ---")
    print(f"Input IDs: {input_ids}")
    print(f"Attention Mask: {attention_mask}")
    print("Taxonomic Levels and Encodings:")
    for level, details in lvl_output.items():
        print(f"  {level.capitalize()}: Label = {details['label']}, Encoded = {details['encoded_label']}")
    
    print("\n--- Output Information ---")
    for lvl, lvl_logits in enumerate(logits):
        predicted_class = lvl_logits.argmax(dim=-1).item()
        print(f"Level {lvl}:")
        print(f"  Logits: {lvl_logits}")
        print(f"  Predicted Class Index: {predicted_class}")


--- Input Information ---
Input IDs: tensor([[ 7, 40, 24, 56, 31, 63, 47, 58, 60, 47, 26, 32, 34, 21, 26, 67, 60, 20,
         33, 25, 64, 62, 68, 46, 15, 42, 32, 35, 21, 27, 38, 30, 63, 45, 34, 42,
         42, 35, 42, 46, 42, 32, 21, 33, 42, 26, 25, 12, 68, 60,  6, 16, 38, 50,
         53, 41,  6, 39,  8,  6, 21,  6, 36, 57,  9, 11, 40, 34, 67, 52, 35, 41,
         62, 19,  7,  6, 41, 14, 37, 19, 29, 55, 53, 62, 37, 66, 13, 20, 27, 39,
         57, 59,  8, 52, 61, 11, 22, 20, 43, 26, 60, 48, 12, 27, 39, 46, 19, 28,
         50, 39, 32, 24, 64, 21, 27, 57, 14, 59, 44, 64, 52, 63, 58, 27, 59, 67,
         34, 23, 31, 31, 10, 29,  7, 17, 55, 41, 39, 60, 51, 13, 26, 61, 62, 14,
         39, 36, 33, 41, 31, 34, 63, 11, 50, 68, 15, 36, 48, 34,  6, 41, 52, 50,
          7, 66, 62,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 